# Ground-truth check — Race Control Observer  *(maintainer only, not for students)*

Run this **before** the first event. It answers four questions that the hack currently
takes on faith:

1. **Which mosaic/panel actually shows each incident?** The repo asserts Günther→`Cam05`
   and Fenestraz→`Cam07` (both in `grp_02`). Track geometry says both cars stopped in the
   **T12–T14** region, which is `grp_05` (`Cam17–Cam20`). One of those is wrong.
2. **Does the naive prompt *fail* the way we want to teach?** The restructured Task 2 needs
   *"is there a stopped car?"* to say **yes** on a car that actually spun and drove away.
   If it doesn't false-positive, the planned lesson has no failure to hang on.
3. **What is the six-group sweep really seeing at 1780?** `#23 Fenestraz` is stranded from
   1692 to 2694. `_aggregate` is *any group blocked → blocked*, so the `--at 1780 --cars 48`
   acceptance test may be corroborating **#23**, not **#48**.
4. **How repeatable is the Günther verdict?** One "not blocked" run needs a rate, not an anecdote.

Everything writes to `groundtruth_summary.json` at the end — send that back along with any
contact sheets that look surprising.

**Runtime:** ~10 min. **Cost:** ~70 Gemini calls on 60s clips.

## 0 · Setup

In [ ]:
# Colab: authenticate so gcloud/storage reads work with YOUR identity.
try:
    from google.colab import auth as _colab_auth
    _colab_auth.authenticate_user()
    print("Colab auth OK")
except Exception as e:
    print("not Colab (or already authed):", e)

import subprocess, sys
def pipq(*pkgs):
    subprocess.run([sys.executable,"-m","pip","install","-q",*pkgs], check=True)
try:
    import cv2
except Exception:
    pipq("opencv-python-headless"); import cv2
try:
    from google import genai
    from google.genai import types
except Exception:
    pipq("google-genai")
    from google import genai
    from google.genai import types
print("deps ready")

In [ ]:
import os, json, gzip, math, time, urllib.request, statistics
from datetime import datetime, timedelta, timezone
import matplotlib.pyplot as plt

# ---- PROJECT -------------------------------------------------------------
PROJECT_ID = ""   # <<< SET THIS to your lab/GCP project id. Colab auto-detect
                  # <<< usually FAILS or picks a personal project. Do not skip.
if not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT","") or ""
if not PROJECT_ID:
    try:
        import google.auth; PROJECT_ID = google.auth.default()[1] or ""
    except Exception: pass
if not PROJECT_ID:
    PROJECT_ID = subprocess.run(["gcloud","config","get-value","project"],
                                capture_output=True, text=True).stdout.strip()
assert PROJECT_ID, "Set PROJECT_ID manually above."

# ---- MOSAICS -------------------------------------------------------------
# Gemini reads gs:// as the VERTEX AI SERVICE AGENT of PROJECT_ID, not as you.
# So GEMINI_BASE must be a bucket that agent can read -> normally the staged
# copy in your own project. Frame extraction below uses YOUR creds, so it can
# read class-demo directly if you prefer.
GEMINI_BASE = f"gs://{PROJECT_ID}-fe-mosaics/mosaics"
FRAME_BASE  = GEMINI_BASE            # or "gs://class-demo/formula-e/r10/mosaics"

GEM_MODEL = "gemini-3.5-flash"
LEAD_S, TAIL_S = 10, 50
GREEN_UTC = datetime(2024, 5, 12, 13, 4, 0, tzinfo=timezone.utc)

GROUPS = {
    "grp_01_cam01_cam02_cam03_cam04": ["Cam01","Cam02","Cam03","Cam04"],
    "grp_02_cam05_cam06_cam07_cam08": ["Cam05","Cam06","Cam07","Cam08"],
    "grp_03_cam09_cam10_cam11_cam12": ["Cam09","Cam10","Cam11","Cam12"],
    "grp_04_cam13_cam14_cam15_cam16": ["Cam13","Cam14","Cam15","Cam16"],
    "grp_05_cam17_cam18_cam19_cam20": ["Cam17","Cam18","Cam19","Cam20"],
    "grp_06_cam21_cam22_cam23_cam24": ["Cam21","Cam22","Cam23","Cam24"],
}
PANELS = ["TL","TR","BL","BR"]

RESULTS = {"project": PROJECT_ID, "model": GEM_MODEL,
           "gemini_base": GEMINI_BASE, "frame_base": FRAME_BASE}
print("project:", PROJECT_ID)
print("gemini reads:", GEMINI_BASE)
print("frames from :", FRAME_BASE)

## 1 · Telemetry ground truth — where and when every car actually stopped

No model involved. This is the physical truth the rest of the notebook is checked against.

In [ ]:
FRAMES = "frames.jsonl.gz"
RAW = ("https://raw.githubusercontent.com/haggman/"
       "formula-e-race-control-observer/main/simulator/src/frames.jsonl.gz")
if not os.path.exists(FRAMES):
    urllib.request.urlretrieve(RAW, FRAMES)

series, drivers = {}, {}
with gzip.open(FRAMES,"rt") as f:
    for line in f:
        o = json.loads(line); t = o["race_time_s"]
        for c in o["cars"]:
            series.setdefault(c["car_number"],{})[t] = c
            drivers[c["car_number"]] = c["driver_name"]

# the detector's own pit-lane guard, copied from observers/telemetry/detector.py
PIT_BOX = {"lat_min":52.4797,"lat_max":52.4806,"lng_min":13.3916,"lng_max":13.3938}
def in_pit(la,ln):
    b=PIT_BOX
    return b["lat_min"]<=la<=b["lat_max"] and b["lng_min"]<=ln<=b["lng_max"]

M_LAT = 111320.0
M_LNG = 111320.0*math.cos(math.radians(52.48))
def metres(a,b): return math.hypot((a[0]-b[0])*M_LAT,(a[1]-b[1])*M_LNG)

def stops(n, thresh=5.0, hold=6):
    out, run = [], []
    for t in sorted(series[n]):
        if series[n][t]["speed_kmh"] <= thresh: run.append(t)
        else:
            if len(run) >= hold: out.append((run[0], run[-1]))
            run = []
    if len(run) >= hold: out.append((run[0], run[-1]))
    return out

print(f"{'car':>4}  {'driver':22} {'start':>6} {'dur':>5}  {'lat,lng':24} {'pit?':8} recovers?")
print("-"*92)
stop_table = []
for n in sorted(series):
    for a,b in stops(n):
        if a < 30: continue
        c = series[n][a]
        fut = [series[n][x]["speed_kmh"] for x in range(b, b+150) if x in series[n]]
        rec = max(fut) if fut else 0
        row = dict(car=n, driver=drivers[n], start=a, dur=b-a, lat=c["lat"], lng=c["lng"],
                   pit=in_pit(c["lat"],c["lng"]), retired=series[n][b]["is_retired"],
                   recovers_to=round(rec))
        stop_table.append(row)
        print(f"{n:>4}  {drivers[n]:22} {a:>6} {b-a:>4}s  "
              f"{c['lat']:.6f},{c['lng']:.6f}  {'PIT-BOX' if row['pit'] else 'ON TRACK':8} "
              f"{'RETIRED' if row['retired'] else f'yes -> {rec:.0f} km/h'}")
RESULTS["stops"] = stop_table

In [ ]:
# How far apart are the incidents? (Do two 'separate' incidents share a camera?)
key = {f"#{r['car']} @{r['start']}": (r["lat"], r["lng"]) for r in stop_table}
names = list(key)
print("pairwise distance, metres\n")
print(" "*22 + "".join(f"{n:>18}" for n in names))
for a in names:
    print(f"{a:22}" + "".join(f"{metres(key[a],key[b]):>18.0f}" for b in names))

## 2 · Geometry cross-reference — which camera *should* see each stop

`frontend/static/track_geometry.json` places all 24 cameras on the circuit outline, and
`prelab/camera_order.txt` labels each with its turn. Inverting the SVG projection gives an
approximate lat/lng per camera.

**Caveat:** these x,y may have been placed by eye for the UI rather than surveyed. Treat the
result as a *hypothesis to check against section 3's images*, not as truth. What matters is
whether it agrees with the pictures.

In [ ]:
GEO_URL = ("https://raw.githubusercontent.com/haggman/"
           "formula-e-race-control-observer/main/frontend/static/track_geometry.json")
geo = json.loads(urllib.request.urlopen(GEO_URL).read().decode())
b = geo["bounds"]; W,H,pad = b["w"], b["h"], b["pad"]

def xy2ll(x,y):
    lng = b["lng_min"] + (x-pad)/(W-2*pad)*(b["lng_max"]-b["lng_min"])
    lat = b["lat_max"] - (y-pad)/(H-2*pad)*(b["lat_max"]-b["lat_min"])
    return lat,lng

group_of = {c:g for g,cs in GROUPS.items() for c in cs}
cams = [(c["id"], c.get("turn",""), *xy2ll(c["x"], c["y"])) for c in geo["cameras"]]

print("nearest cameras to each stop (derived positions — verify against the images!)\n")
nearest = {}
for r in stop_table:
    p = (r["lat"], r["lng"])
    top = sorted((metres(p,(c[2],c[3])), c[0], c[1]) for c in cams)[:3]
    nearest[f"#{r['car']} @{r['start']}"] = [
        {"camera":cid, "turn":turn, "metres":round(d), "group":group_of[cid]}
        for d,cid,turn in top]
    print(f"  #{r['car']:<3} @{r['start']:<5} -> " +
          ",  ".join(f"{cid} ({turn}) {d:.0f}m  [{group_of[cid][:6]}]" for d,cid,turn in top))
RESULTS["nearest_cameras"] = nearest

print("\nWhat the repo currently ASSERTS:")
print("  Günther   @693  -> Cam05  (grp_02)   [STUDENT_GUIDE acceptance test + notebook GUNTHER_GROUP]")
print("  Fenestraz @1698 -> Cam07  (grp_02)   [STUDENT_GUIDE acceptance test + track_geometry note]")
print("  Mortara   @1780 -> Cam07  (grp_02)   [STUDENT_GUIDE acceptance test]")

## 3 · The contact sheet — look at the footage yourself

**This is the decisive cell.** For each incident it pulls the frame at the *end of the
verification window* (`t + 50`), which is the moment the persistence question is actually
about, from **all six mosaics**, and labels each quadrant with its camera id.

Find the stopped car with your own eyes. Whichever mosaic it's in is the ground truth,
and every camera id in the docs should be reconciled to that.

In [ ]:
def local_mp4(gid):
    """Download one mosaic once. Uses the storage client (always present in Colab);
    falls back to the gcloud CLI if the client isn't available."""
    p = f"{gid}.mp4"
    if os.path.exists(p) and os.path.getsize(p) > 0:
        return p
    src = f"{FRAME_BASE}/{gid}.mp4"
    print(f"  downloading {gid} …")
    try:
        from google.cloud import storage
        rest = FRAME_BASE[len("gs://"):]
        bkt, _, prefix = rest.partition("/")
        client = storage.Client(project=PROJECT_ID)
        client.bucket(bkt).blob(f"{prefix}/{gid}.mp4").download_to_filename(p)
    except Exception as e:
        print(f"    storage client failed ({str(e)[:120]}) — trying gcloud CLI")
        subprocess.run(["gcloud","storage","cp",src,p], check=True)
    return p

def frame_at(gid, offset_s):
    cap = cv2.VideoCapture(local_mp4(gid))
    fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(offset_s*fps))
    ok, img = cap.read(); cap.release()
    if not ok: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for i,(dy,dx) in enumerate([(0,0),(0,1),(1,0),(1,1)]):     # TL TR BL BR
        cam = GROUPS[gid][i]
        cv2.putText(img, f"{PANELS[i]} {cam}", (dx*w//2+14, dy*h//2+46),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0,0,0), 7)
        cv2.putText(img, f"{PANELS[i]} {cam}", (dx*w//2+14, dy*h//2+46),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (255,255,0), 3)
    return img

def contact_sheet(t, label, offset=None):
    off = t + TAIL_S if offset is None else offset
    utc = GREEN_UTC + timedelta(seconds=off)
    print(f"\n=== {label} — race-second {t}, showing offset {off} "
          f"(end of window; wall clock ~{utc:%H:%M:%S} UTC) ===")
    fig, axes = plt.subplots(3, 2, figsize=(22, 20))
    for ax, gid in zip(axes.ravel(), GROUPS):
        img = frame_at(gid, off)
        ax.imshow(img); ax.set_title(gid, fontsize=13); ax.axis("off")
    plt.tight_layout(); plt.show()

def zoom(gid_prefix, t, offset=None):
    """Full-size look at ONE mosaic, e.g. zoom('grp_05', 693)."""
    gid = next(g for g in GROUPS if g.startswith(gid_prefix))
    off = t + TAIL_S if offset is None else offset
    plt.figure(figsize=(20,15)); plt.imshow(frame_at(gid, off)); plt.axis("off")
    plt.title(f"{gid} @ offset {off}", fontsize=15); plt.show()
print("helpers ready")

In [ ]:
contact_sheet(693,  "GÜNTHER #7 — stops 693, retires (expect a PERSISTENT blockage)")

In [ ]:
contact_sheet(1692, "FENESTRAZ #23 + NATO #17 — both stop 1692; #17 leaves ~1701, #23 stays 1002s")

In [ ]:
contact_sheet(1780, "MORTARA #48 — stops 1780, drives away after 18s (expect CLEARED here; "
                    "#23 is still stranded elsewhere on track)")

In [ ]:
contact_sheet(94,   "TICKTUM #33 — pit lane (expect no track blockage anywhere)")

### Optional: the same moments *before* the tail plays

Useful for the "verify too early and you hallucinate" war story — and for seeing Mortara
stopped before he recovers.

In [ ]:
contact_sheet(1780, "MORTARA #48 — at the STOP itself (car should be stationary here)", offset=1785)
# zoom('grp_01', 1780)      # uncomment to inspect one mosaic full-size

## 4 · Naive vs persistence — does the planned Task-2 failure actually happen?

The restructured Task 2 depends on this: the obvious prompt (*"is there a stopped car?"*)
must say **yes** on **Mortara @1780**, a driver who stopped for 18 seconds and then drove
away at 219 km/h. That false positive is the failure students feel before they earn the
END-of-window framing.

Run both prompts on **every** group so we can see which group holds which car.

In [ ]:
GEM = genai.Client(vertexai=True, project=PROJECT_ID, location="global")

NAIVE = """You are a race-control video verifier. Telemetry flagged a car that may be
stopped on the track. Watch the clip and tell me: is there a stopped car on the track?
Say which panel (TL/TR/BL/BR) and the car number if you can read it."""

PERSISTENCE = """You are a race-control video verifier deciding whether a SAFETY CAR is
warranted. Telemetry flagged a car that may be stopped on the track. Watch the whole clip
and judge the TRACK STATE by the END of it (the safety call is about the track, not which car):
- A car STILL stopped/stranded on or beside the racing line at the end (a persistent
  obstruction, maybe with marshals or a recovery vehicle): blockage=true, cleared=false.
- A car appeared but DROVE AWAY / was recovered / the line is clear by the end:
  blockage=false, cleared=true.
- No stopped car at any point: blockage=false, cleared=false.
Base the answer on the END of the window, not a single moment. Report which panel
(TL/TR/BL/BR) the incident is in, the car number only if clearly legible (else null), a
one-line what_you_see, and whether other cars are still moving (feed_live)."""

CONTRACT = ('Respond with a SINGLE JSON object: {"blockage": bool, "cleared": bool, '
            '"panel": "TL|TR|BL|BR|none", "feed_live": bool, '
            '"seen_car": <car number if clearly readable, else null>, '
            '"what_you_see": string, "confidence": number}')

def _parse(txt):
    s = (txt or "").strip(); a,bb = s.find("{"), s.rfind("}")
    if a != -1 and bb > a:
        try: return json.loads(s[a:bb+1])
        except Exception: pass
    return {}

def _truthy(v):
    if isinstance(v, bool): return v
    if isinstance(v,(int,float)): return v != 0
    if isinstance(v,str): return v.strip().lower() in ("true","yes","y","1","blocked","blockage")
    return False

def ask(gid, t, body, lead=LEAD_S, tail=TAIL_S, retries=3):
    start, end = max(0,t-lead), t+tail
    cams = GROUPS[gid]
    ctx = (f"This is a ~{end-start}s CCTV clip — a 2x2 mosaic of four cameras: "
           f"TL={cams[0]}, TR={cams[1]}, BL={cams[2]}, BR={cams[3]}.")
    vpart = types.Part(
        file_data=types.FileData(file_uri=f"{GEMINI_BASE}/{gid}.mp4", mime_type="video/mp4"),
        video_metadata=types.VideoMetadata(start_offset=f"{start}s", end_offset=f"{end}s"))
    for attempt in range(retries):
        try:
            r = GEM.models.generate_content(
                model=GEM_MODEL,
                contents=[types.Content(role="user",
                    parts=[vpart, types.Part(text=f"{body}\n{ctx}\n{CONTRACT}")])],
                config=types.GenerateContentConfig(temperature=0.2,
                                                   response_mime_type="application/json"))
            return _parse(r.text)
        except Exception as e:
            if attempt == retries-1: return {"_error": str(e)[:200]}
            time.sleep(8*(attempt+1))

def compare(t, label):
    print(f"\n### {label}  (race-second {t})")
    print(f"{'group':10} {'prompt':12} {'blockage':>9} {'cleared':>8} {'panel':>6} {'camera':>7} {'conf':>5}  what_you_see")
    print("-"*130)
    rows = []
    for gid in GROUPS:
        for tag, body in (("naive", NAIVE), ("persistence", PERSISTENCE)):
            d = ask(gid, t, body)
            pan = str(d.get("panel","none"))
            cam = GROUPS[gid][PANELS.index(pan)] if pan in PANELS else "-"
            rows.append(dict(t=t, group=gid, prompt=tag, blockage=_truthy(d.get("blockage")),
                             cleared=_truthy(d.get("cleared")), panel=pan, camera=cam,
                             confidence=d.get("confidence"), seen_car=d.get("seen_car"),
                             what_you_see=d.get("what_you_see",""), error=d.get("_error")))
            print(f"{gid[:9]:10} {tag:12} {str(_truthy(d.get('blockage'))):>9} "
                  f"{str(_truthy(d.get('cleared'))):>8} {pan:>6} {cam:>7} "
                  f"{str(d.get('confidence','?')):>5}  {str(d.get('what_you_see',''))[:60]}")
    return rows

cmp_rows  = compare(693,  "GÜNTHER #7 — retires. Want: persistence=BLOCKED")
cmp_rows += compare(1780, "MORTARA #48 — recovers. Want: naive says STOPPED (the teaching "
                          "failure), persistence says CLEARED, in #48's own group")
RESULTS["naive_vs_persistence"] = cmp_rows

## 5 · Full sweep — what does *any-group-blocked* actually corroborate?

`_aggregate` returns `blocked` if **any** of the six groups reports a blockage, with no
locality gate. `#23` is stranded from 1692 to 2694, so every sweep in that range can be
corroborated by `#23` regardless of which car telemetry flagged.

This prints per-group verdicts so we can see *which* group carries each verdict.

In [ ]:
def sweep(t, label):
    print(f"\n### {label}  (race-second {t})")
    print(f"{'group':32} {'state':9} {'panel':>6} {'camera':>7} {'conf':>5}  what_you_see")
    print("-"*130)
    rows, blocked, cleared = [], [], []
    for gid in GROUPS:
        d = ask(gid, t, PERSISTENCE)
        pan = str(d.get("panel","none"))
        cam = GROUPS[gid][PANELS.index(pan)] if pan in PANELS else None
        st = "BLOCKED" if _truthy(d.get("blockage")) else ("cleared" if _truthy(d.get("cleared")) else "-")
        if st=="BLOCKED": blocked.append((cam,d))
        elif st=="cleared": cleared.append((cam,d))
        rows.append(dict(t=t, group=gid, state=st, panel=pan, camera=cam,
                         confidence=d.get("confidence"), seen_car=d.get("seen_car"),
                         what_you_see=d.get("what_you_see",""), error=d.get("_error")))
        print(f"{gid:32} {st:9} {pan:>6} {str(cam):>7} {str(d.get('confidence','?')):>5}  "
              f"{str(d.get('what_you_see',''))[:60]}")
    if blocked:
        cam,d = max(blocked, key=lambda x: x[1].get("confidence",0) or 0)
        verdict = f"blocked ({cam})"
    elif cleared: verdict = "cleared"
    else: verdict = "unseen"
    print(f"  => AGGREGATE VERDICT: {verdict}")
    return rows, verdict

sw = {}
for t,label,expect in [(693,"GÜNTHER #7","repo expects blocked/Cam05"),
                       (1698,"FENESTRAZ #23 + NATO #17","repo expects blocked/Cam07"),
                       (1780,"MORTARA #48","repo expects blocked/Cam07 — but #48 recovers"),
                       (94,  "TICKTUM #33 (pit)","expect no track blockage")]:
    rows, verdict = sweep(t, f"{label} — {expect}")
    sw[t] = {"rows": rows, "verdict": verdict, "repo_expectation": expect}
RESULTS["sweeps"] = sw

## 6 · Repeatability — is "not blocked" a rate or an anecdote?

Same group, same second, same prompt, N times. If Günther flips between runs, the fix is
prompt/window robustness. If it's stable-but-wrong, the fix is the camera mapping.

In [ ]:
REPS = 5
def repeat(gid_prefix, t, body, tag, reps=REPS):
    gid = next(g for g in GROUPS if g.startswith(gid_prefix))
    out = []
    for i in range(reps):
        d = ask(gid, t, body)
        out.append(dict(blockage=_truthy(d.get("blockage")), cleared=_truthy(d.get("cleared")),
                        panel=d.get("panel"), confidence=d.get("confidence"),
                        what_you_see=d.get("what_you_see","")))
        print(f"  {tag} {gid[:6]} @{t} run {i+1}/{reps}: blockage={out[-1]['blockage']} "
              f"cleared={out[-1]['cleared']} panel={out[-1]['panel']} "
              f"— {str(out[-1]['what_you_see'])[:70]}")
    n = sum(1 for r in out if r["blockage"])
    print(f"  => {tag}: blocked {n}/{reps}\n")
    return {"group": gid, "t": t, "runs": out, "blocked_rate": f"{n}/{reps}"}

rep = {}
# the group the repo hardcodes as GUNTHER_GROUP:
rep["gunther_grp02"] = repeat("grp_02", 693, PERSISTENCE, "grp_02 (repo's claim)")
# the group the geometry says should hold Günther:
rep["gunther_grp05"] = repeat("grp_05", 693, PERSISTENCE, "grp_05 (geometry's claim)")
RESULTS["repeatability"] = rep

## 7 · Summary — send this back

In [ ]:
RESULTS["generated_utc"] = datetime.now(timezone.utc).isoformat()
with open("groundtruth_summary.json","w") as fh:
    json.dump(RESULTS, fh, indent=2, default=str)

print("="*78)
print("HEADLINES")
print("="*78)
for t, s in RESULTS.get("sweeps", {}).items():
    print(f"  t={t:<5} aggregate={s['verdict']:<18} ({s['repo_expectation']})")
    for r in s["rows"]:
        if r["state"] != "-":
            print(f"          {r['group'][:6]} {r['state']:8} cam={r['camera']} "
                  f"— {str(r['what_you_see'])[:70]}")
print()
for k,v in RESULTS.get("repeatability", {}).items():
    print(f"  {k}: blocked {v['blocked_rate']}")
print()
naive_1780 = [r for r in RESULTS.get("naive_vs_persistence",[])
              if r["t"]==1780 and r["prompt"]=="naive" and r["blockage"]]
pers_1780  = [r for r in RESULTS.get("naive_vs_persistence",[])
              if r["t"]==1780 and r["prompt"]=="persistence" and r["cleared"]]
print(f"  Mortara @1780: naive claimed a blockage in {len(naive_1780)} group(s); "
      f"persistence said cleared in {len(pers_1780)} group(s).")
print("  -> the planned Task-2 failure is REAL if naive > 0 and persistence flips it.")
print()
print("wrote groundtruth_summary.json")
try:
    from google.colab import files; files.download("groundtruth_summary.json")
except Exception: pass